## 09 — Evaluation & Comparison: Paper Models vs NLP Models (Milestone 2)

This notebook loads results from all 8 models, computes the two required metrics,
and saves a comparison table for use by the plot generation notebook.

### Inputs
**Paper models** (from `04_models_paper.ipynb`, saved in `../data/processed/`):
- `model1_fft_mlp_all_outputs.pt`
- `model2_fft_mlp_imu_thm_outputs.pt`
- `model3_cnn_bilstm_tof_outputs.pt`
- `model4_late_fusion_outputs.pt`
- `model5_intermediate_fusion_outputs.pt`
- `model6_fft_random_forest_outputs.pt`

**NLP models** (from `07` and `08`, saved in `../results/`):
- `transformer_results.pkl`
- `bilstm_results.pkl`

### Outputs
- `../results/all_model_comparison.csv` — full comparison table for plot generation

## 1. Imports

Importing the libraries needed to load saved model outputs, compute evaluation metrics,
and assemble the final comparison table.

In [13]:
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import f1_score

DATA_PROCESSED = Path("../data/processed")
RESULTS_DIR    = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports done.")

Imports done.


## 2. BFRB Gesture Definition

Defining the gesture labels that count as BFRB-related so every model is evaluated
with the same binary grouping.

In [14]:
BFRB_GESTURES = {
    'Cheek - pinch skin',
    'Forehead - pull hairline',
    'Neck - scratch',
    'Neck - pinch skin',
    'Eyelash - pull hair',
    'Eyebrow - pull hair',
    'Forehead - scratch',
    'Above ear - pull hair',
    'Pinch knee/leg skin',
    'Scratch knee/leg skin',
}

print(f"BFRB gesture count: {len(BFRB_GESTURES)}")

BFRB gesture count: 10


## 3. Metrics Function

This helper computes the two project metrics used throughout the notebook:
binary F1 for BFRB vs non-BFRB and macro F1 across all gesture classes.

In [15]:
def compute_metrics(outputs, bfrb_gestures):
    """
    Compute binary F1 and macro F1 from a paper model output dict.
    Reused from 05_evaluation.ipynb.
    """
    y_true  = outputs["y_true"].cpu().numpy()
    y_pred  = outputs["y_pred"].cpu().numpy()
    classes = [str(c) for c in outputs["classes"]]

    # Map integer IDs back to gesture names
    y_true_g = np.array([classes[i] for i in y_true], dtype=object)
    y_pred_g = np.array([classes[i] for i in y_pred], dtype=object)

    # Binary F1: BFRB=1, non-BFRB=0
    y_true_bin = np.array([1 if g in bfrb_gestures else 0 for g in y_true_g])
    y_pred_bin = np.array([1 if g in bfrb_gestures else 0 for g in y_pred_g])
    binary_f1  = float(f1_score(y_true_bin, y_pred_bin, average="binary", pos_label=1))

    # Macro F1: across all 18 gesture classes
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    return binary_f1, macro_f1


print("compute_metrics defined.")

compute_metrics defined.


## 4. Load Paper Model Results

Loading the saved output files from the six baseline paper models and computing
their binary and macro F1 scores using the shared evaluation function.

In [16]:
paper_model_files = {
    "FFT-MLP (All Sensors)" : "model1_fft_mlp_all_outputs.pt",
    "FFT-MLP (IMU+THM)"     : "model2_fft_mlp_imu_thm_outputs.pt",
    "CNN-BiLSTM (TOF)"      : "model3_cnn_bilstm_tof_outputs.pt",
    "Late Fusion"           : "model4_late_fusion_outputs.pt",
    "Intermediate Fusion"   : "model5_intermediate_fusion_outputs.pt",
    "FFT-Random Forest"     : "model6_fft_random_forest_outputs.pt",
}

paper_results = []
for model_name, filename in paper_model_files.items():
    outputs   = torch.load(DATA_PROCESSED / filename, weights_only=False)
    binary_f1, macro_f1 = compute_metrics(outputs, BFRB_GESTURES)
    paper_results.append({
        "model"     : model_name,
        "type"      : "Paper (Baseline)",
        "binary_f1" : binary_f1,
        "macro_f1"  : macro_f1
    })
    print(f"{model_name:<25} | Binary F1: {binary_f1*100:.2f}% | Macro F1: {macro_f1*100:.2f}%")

FFT-MLP (All Sensors)     | Binary F1: 90.93% | Macro F1: 46.12%
FFT-MLP (IMU+THM)         | Binary F1: 90.86% | Macro F1: 41.07%
CNN-BiLSTM (TOF)          | Binary F1: 91.60% | Macro F1: 29.50%
Late Fusion               | Binary F1: 94.28% | Macro F1: 50.14%
Intermediate Fusion       | Binary F1: 91.11% | Macro F1: 39.61%
FFT-Random Forest         | Binary F1: 89.81% | Macro F1: 38.80%


## 5. Load NLP Model Results

Loading the saved results from the Transformer and BiLSTM NLP models so they can
be compared directly against the baseline paper models.

In [17]:
nlp_model_files = {
    "Transformer (NLP)" : "transformer_results.pkl",
    "BiLSTM (NLP)"      : "bilstm_results.pkl",
}

nlp_results = []
for model_name, filename in nlp_model_files.items():
    with open(RESULTS_DIR / filename, "rb") as f:
        res = pickle.load(f)
    nlp_results.append({
        "model"     : model_name,
        "type"      : "NLP (Novel)",
        "binary_f1" : res["binary_f1"],
        "macro_f1"  : res["macro_f1"]
    })
    print(f"{model_name:<25} | Binary F1: {res['binary_f1']*100:.2f}% | Macro F1: {res['macro_f1']*100:.2f}%")

Transformer (NLP)         | Binary F1: 90.93% | Macro F1: 85.98%
BiLSTM (NLP)              | Binary F1: 89.98% | Macro F1: 84.19%


## 6. Build and Save Comparison Table

Combining all model results into one table, formatting the scores as percentages,
and saving the final comparison file for downstream visualization.

In [18]:
all_results = pd.DataFrame(paper_results + nlp_results)
all_results["binary_f1_pct"] = (all_results["binary_f1"] * 100).round(2)
all_results["macro_f1_pct"]  = (all_results["macro_f1"]  * 100).round(2)

print("=== Full Model Comparison ===")
print(all_results[["model", "type", "binary_f1_pct", "macro_f1_pct"]].to_string(index=False))

all_results.to_csv(RESULTS_DIR / "all_model_comparison.csv", index=False)
print("\nSaved: ../results/all_model_comparison.csv")

best_binary = all_results.loc[all_results["binary_f1_pct"].idxmax()]
best_macro  = all_results.loc[all_results["macro_f1_pct"].idxmax()]
print(f"\nBest Binary F1 : {best_binary['model']} ({best_binary['binary_f1_pct']:.2f}%)")
print(f"Best Macro F1  : {best_macro['model']} ({best_macro['macro_f1_pct']:.2f}%)")

=== Full Model Comparison ===
                model             type  binary_f1_pct  macro_f1_pct
FFT-MLP (All Sensors) Paper (Baseline)          90.93         46.12
    FFT-MLP (IMU+THM) Paper (Baseline)          90.86         41.07
     CNN-BiLSTM (TOF) Paper (Baseline)          91.60         29.50
          Late Fusion Paper (Baseline)          94.28         50.14
  Intermediate Fusion Paper (Baseline)          91.11         39.61
    FFT-Random Forest Paper (Baseline)          89.81         38.80
    Transformer (NLP)      NLP (Novel)          90.93         85.98
         BiLSTM (NLP)      NLP (Novel)          89.98         84.19

Saved: ../results/all_model_comparison.csv

Best Binary F1 : Late Fusion (94.28%)
Best Macro F1  : Transformer (NLP) (85.98%)
